In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time


In [ ]:

# =========================
# EDIT THESE SETTINGS
# =========================
SEARCH_QUERY = "wireless headphones"   # change this to your category
NUM_PAGES = 3                          # how many Amazon result pages to scrape
BASE_URL = "https://www.amazon.com/s"
OUTPUT_FILE = "amazon_products.csv"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# =========================
# HELPER FUNCTIONS
# =========================
def clean_price(text):
    if not text:
        return None
    match = re.search(r"[\d,]+\.\d{2}", text)
    if match:
        return float(match.group().replace(",", ""))
    return None

def clean_rating(text):
    if not text:
        return None
    match = re.search(r"\d+\.\d+", text)
    if match:
        return float(match.group())
    return None

def clean_reviews(text):
    if not text:
        return None
    text = text.replace(",", "").strip()
    if text.isdigit():
        return int(text)
    return None

# =========================
# MAIN SCRAPING LOOP
# =========================
all_products = []

for page in range(1, NUM_PAGES + 1):
    params = {
        "k": SEARCH_QUERY,
        "page": page
    }

    response = requests.get(BASE_URL, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    product_cards = soup.select('div[data-component-type="s-search-result"]')

    for item in product_cards:
        # EDIT THESE SELECTORS IF NEEDED
        title_tag = item.select_one("h2 span")
        link_tag = item.select_one("h2 a")
        price_tag = item.select_one(".a-price .a-offscreen")
        rating_tag = item.select_one(".a-icon-alt")

        # Review count on Amazon can vary by layout
        review_tag = item.select_one("span.a-size-base")

        title = title_tag.get_text(strip=True) if title_tag else None
        link = "https://www.amazon.com" + link_tag["href"] if link_tag and link_tag.get("href") else None
        price = clean_price(price_tag.get_text(strip=True) if price_tag else None)
        rating = clean_rating(rating_tag.get_text(strip=True) if rating_tag else None)

        review_count = None
        if review_tag:
            review_count = clean_reviews(review_tag.get_text(strip=True))

        all_products.append({
            "title": title,
            "price": price,
            "rating": rating,
            "review_count": review_count,
            "link": link,
            "search_query": SEARCH_QUERY,
            "source": "Amazon"
        })

    time.sleep(2)  # be polite and reduce the chance of blocking

# =========================
# CLEAN INTO DATAFRAME
# =========================
df = pd.DataFrame(all_products)

# Remove rows missing the key fields you need
df = df.dropna(subset=["title", "price", "rating"])

# Optional: remove duplicates
df = df.drop_duplicates(subset=["title", "price", "rating"])

# Save
df.to_csv(OUTPUT_FILE, index=False)

print(df.head())
print(f"Saved {len(df)} rows to {OUTPUT_FILE}")

KeyError: ['title', 'price', 'rating']